## Importeren
importeren van paketten

In [ ]:
# --- Standard library ---
import json
import math
import os
import shutil
import warnings
import zipfile
from collections import deque
from collections.abc import Callable
from dataclasses import dataclass, field
from enum import Enum
from functools import cache
from pathlib import Path

import matplotlib.pyplot as plt

# --- Third-party ---
import numpy as np
import pandas as pd
import shapely
from geolib.geometry import Point as glPoint
from geolib.models.dstability.analysis import (
    DStabilityCircle,
    DStabilitySearchArea,
    DStabilitySlipPlaneConstraints,
    DStabilityUpliftVanAnalysisMethod,
    DStabilityUpliftVanParticleSwarmAnalysisMethod,
)

# --- Project imports ---
from geolib.models.dstability.dstability_model import DStabilityModel
from geolib.models.dstability.internal import CalculationTypeEnum, PersistableSoilVisualization, Stage
from geolib.models.dstability.internal import PersistableShadingTypeEnum as PSE
from geolib.models.dstability.states import DStabilityStatePoint
from matplotlib.patches import Patch
from shapely import (
    GeometryCollection,
    MultiLineString,
    MultiPolygon,
    polygonize,
    set_precision,
    snap,
    split,
    unary_union,
)
from shapely.geometry import LineString, Point, Polygon
from src.dstab.boreholes.borehole import Borehole, BoreholeLayer  # pyright: ignore[reportMissingImports]
from src.dstab.geometry.geom_utils import (  # pyright: ignore[reportMissingImports]
    add_point_to_polygon,
    interpolate_coordinates,
)
from src.dstab.geometry.surface_line import CPoint, SurfaceLine  # pyright: ignore[reportMissingImports]
from src.dstab.geometry.waternet import HeadLine, ReferenceLine  # pyright: ignore[reportMissingImports]
from src.dstab.model.settings import Calculation, Scenario, UpliftSettings  # pyright: ignore[reportMissingImports]
from src.dstab.parsers.soil_from_excel import parse_from_excel  # pyright: ignore[reportMissingImports]
from src.dstab.plotting.plot_utils import plot_polygon  # pyright: ignore[reportMissingImports]
from src.dstab.soils.soil import ShearStrengthModel, SoilStore  # pyright: ignore[reportMissingImports]
from src.globs import DECIMALS  # pyright: ignore[reportMissingImports]

# --- Warnings ---
warnings.filterwarnings("ignore", category=FutureWarning)

## Locaties

In [ ]:
# locatie templates waarvoor stix bestanden moeten worden gemaakt
loc_templates = r"P:\PR\5536.10\Uitvoering\04 - Fase 1\04 - Analyse\03_Logboeken\test"
# locatie van het grond bestand
path_materials = r"P:\PR\5536.10\Uitvoering\04 - Fase 1\04 - Analyse\05_dstab\Python automatisering\soil.xlsx"
# uitvoer locatie waar stix geplaatst wordt
uitvoer = r"P:\PR\5536.10\Uitvoering\04 - Fase 1\04 - Analyse\05_dstab\Python automatisering\stix_bestanden"

## Read_excel_template

In [ ]:
def get_model_param(df: pd.DataFrame, param: str) -> float:
    value = df[df["Parameter"] == param]["Waarde"].iloc[0]
    if np.isnan(value):
        return np.nan
    return float(value)


def get_uplift_setting(df: pd.DataFrame, label: str):
    return df[df[df.columns[0]] == label][df.columns[3]].values[0]


# noinspection PyTypeChecker
def parse_excel_template(excel_path: str) -> dict:
    if not os.path.exists(excel_path):
        raise FileNotFoundError("Excel file not found")

    excel = pd.ExcelFile(
        excel_path,
        engine="openpyxl",
    )

    """
    Lees het maaiveldprofiel uit
    """
    # Blad naam
    sheet = "Profiel"

    points = pd.read_excel(
        excel,
        sheet_name=sheet,
        usecols=[0, 3],
        skiprows=1,
    ).dropna(subset="x")

    # Punten van het maaiveldprofiel
    x = points["x"]
    z = points["y met bodemdaling"]

    # Kenmerkende punten
    characteristics = pd.read_excel(excel, sheet_name=sheet, usecols="F:H", skiprows=3, nrows=8)

    key_label = "Kenmerkend punt"

    points = [CPoint(x=r["x"], z=r["y"], label=r[key_label]) for _, r in characteristics.iterrows()]

    sl = SurfaceLine(x=x, z=z, characteristic_points=points)

    ######################
    ##  Boorprofielen   ##
    ######################

    sheet = "Ondergrondschematisatie"
    boreholes = []
    for label, range, dist_col in zip(["Buitendijks", "Kruin", "Binnendijks"], ["A:B", "D:E", "G:H"], ["B", "E", "H"]):
        df = pd.read_excel(
            excel,
            sheet_name=sheet,
            usecols=range,
            header=10,
            skiprows=[11],
            nrows=12,
        )
        df.columns = df.columns.str.split(".").str[0]
        x_dist = pd.read_excel(excel, sheet_name=sheet, usecols=dist_col, header=8, nrows=1)

        layers = []
        for key, row in df.iterrows():
            if pd.isnull(row["laag:"]):
                continue
            layers.append(
                BoreholeLayer(code=row["laag:"], name=row["laag:"], top=row["bk grondlaag"])
            )  # ,aquifer_ids = a_ids)

        x_distance = float(x_dist.iloc[0])
        borehole = Borehole(name=label, x_distance=x_distance, layers=layers)
        boreholes.append(borehole)
    ###############################################
    # Waterspanningen freatisch
    ###############################################
    sheet = "Waterspanningen"
    df = pd.read_excel(excel, sheet_name=sheet, usecols="B:C", header=13, nrows=9)
    df.columns = df.columns.str.split(".").str[0]

    x = list(df["x"])
    z = list(df["y"])
    daily_headline = HeadLine(
        x=x,
        z=z,
        label="Dagelijkse grondwaterstand",
        is_phreatic=True,
    )

    sheet = "Waterspanningen"
    df = pd.read_excel(excel, sheet_name=sheet, usecols="E:P", header=13, nrows=9)
    df.columns = df.columns.str.split(".").str[0]

    fr_1 = HeadLine(
        x=list(df.iloc[:, 0]),
        z=list(df.iloc[:, 1]),
        label="Freatische lijn stage 1",
        is_phreatic=True,
    )
    fr_2 = HeadLine(
        x=list(df.iloc[:, 2]),
        z=list(df.iloc[:, 3]),
        label="Freatische lijn stage 2",
        is_phreatic=True,
    )
    fr_3 = HeadLine(
        x=list(df.iloc[:, 4]),
        z=list(df.iloc[:, 5]),
        label="Freatische lijn stage 3",
        is_phreatic=True,
    )
    fr_4 = HeadLine(
        x=list(df.iloc[:, 6]),
        z=list(df.iloc[:, 7]),
        label="Freatische lijn stage 4",
        is_phreatic=True,
    )
    fr_5 = HeadLine(
        x=list(df.iloc[:, 8]),
        z=list(df.iloc[:, 9]),
        label="Freatische lijn stage 5",
        is_phreatic=True,
    )
    fr_overslag = HeadLine(
        x=list(df.iloc[:, 10]),
        z=list(df.iloc[:, 11]),
        label="Freatische lijn bij overslag",
        is_phreatic=True,
    )

    ###############################################
    # Waterspanningen stijghoogte DL
    ###############################################
    sheet = "Waterspanningen"
    df = pd.read_excel(excel, sheet_name=sheet, usecols="B:C", header=59, nrows=12)
    df.columns = df.columns.str.split(".").str[0]
    x = list(df["x"])
    z = list(df["y"])
    DL_stijg = HeadLine(
        x=x,
        z=z,
        label="Dagelijkse stijghoogte",
        is_phreatic=False,
    )

    sheet = "Waterspanningen"
    df = pd.read_excel(excel, sheet_name=sheet, usecols="D:O", header=59, nrows=12)
    df.columns = df.columns.str.split(".").str[0]
    stijg_1 = HeadLine(
        x=list(df.iloc[:, 0]),
        z=list(df.iloc[:, 1]),
        label="Stijghoogte stage 1",
        is_phreatic=False,
    )
    stijg_2 = HeadLine(
        x=list(df.iloc[:, 2]),
        z=list(df.iloc[:, 3]),
        label="Stijghoogte stage 2",
        is_phreatic=False,
    )
    stijg_3 = HeadLine(
        x=list(df.iloc[:, 4]),
        z=list(df.iloc[:, 5]),
        label="Stijghoogte stage 3",
        is_phreatic=False,
    )
    stijg_4 = HeadLine(
        x=list(df.iloc[:, 6]),
        z=list(df.iloc[:, 7]),
        label="Stijghoogte stage 4",
        is_phreatic=False,
    )
    stijg_5 = HeadLine(
        x=list(df.iloc[:, 8]),
        z=list(df.iloc[:, 9]),
        label="Stijghoogte stage 5",
        is_phreatic=False,
    )
    stijg_overslag = HeadLine(
        x=list(df.iloc[:, 10]),
        z=list(df.iloc[:, 11]),
        label="Stijghoogte bij overslag",
        is_phreatic=False,
    )

    ###############################################
    ## Referentielijnen freatisch
    ###############################################
    sheet = "Waterspanningen"
    df = pd.read_excel(excel, sheet_name=sheet, usecols="A:B", header=84, nrows=531).dropna(subset="x")

    df.columns = df.columns.str.split(".").str[0]
    x = list(df["x"])
    z = list(df["y"])
    ref_line_frea = ReferenceLine(
        x=x,
        z=z,
        label="Overgang freatisch-stijghoogte",
        notes="Boven deze lijn freatisch, onder de lijn interpoleren naar stijghoogte",
    )

    ###############################################
    ## Referentielijnen stijghoogte
    ###############################################
    sheet = "Waterspanningen"
    df = pd.read_excel(excel, sheet_name=sheet, usecols="C:D", header=84, nrows=6)

    df.columns = df.columns.str.split(".").str[0]
    x = list(df["x"])
    z = list(df["y"])
    ref_line_stijg = ReferenceLine(
        x=x, z=z, label="stijghoogte", notes="Boven deze lijn freatisch, onder de lijn stijghoogte"
    )

    ###############################################
    ## Referentielijnen indringing
    ###############################################
    sheet = "Waterspanningen"
    df = pd.read_excel(excel, sheet_name=sheet, usecols="E:F", header=84, nrows=6)

    df.columns = df.columns.str.split(".").str[0]
    x = list(df["x"])
    z = list(df["y"])
    ref_line_indringing = ReferenceLine(
        x=x, z=z, label="stijghoogte", notes="dagelijkse stijghoogte, onder deze lijn hoogwater stijghoogte"
    )

    indringing = list(pd.read_excel(excel, sheet_name=sheet, usecols="B", header=80, nrows=1).iloc[:, 0])

    ###############################################
    ## Referentielijnen overslag
    ###############################################
    sheet = "Waterspanningen"
    df = pd.read_excel(excel, sheet_name=sheet, usecols="G:H", header=84, nrows=531)
    df.columns = df.columns.str.split(".").str[0]
    df = df.dropna(subset="x")
    x = list(df["x"])
    z = list(df["y"])

    ref_line_overslag = ReferenceLine(
        x=x,
        z=z,
        label="freatische lijn overslag",
        notes="Boven deze lijn freatisch tijdens overslag, onder de lijn freatisch",
    )

    ###############################################
    ## Constraint zone A glijcirkel zonder reststerkte
    ###############################################

    sheet = "Rekeninstellingen"
    df = pd.read_excel(excel, sheet_name=sheet, usecols="B:H", header=29, nrows=11)

    constraint_breedte = list(df.loc[1])
    start_constraint_zone = list(df.loc[0])
    return {
        "SurfaceLine": sl,
        "Boreholes": boreholes,
        "DL_FREA": daily_headline,
        "DL_STIJG": DL_stijg,
        "RL_FREA": ref_line_frea,
        "RL_STIJG": ref_line_stijg,
        "RL_INDRINGING": ref_line_indringing,
        "RL_OVERSLAG": ref_line_overslag,
        "FREA_1": fr_1,
        "FREA_2": fr_2,
        "FREA_3": fr_3,
        "FREA_4": fr_4,
        "FREA_5": fr_5,
        "FREA_OVERSLAG": fr_overslag,
        "STIJG_1": stijg_1,
        "STIJG_2": stijg_2,
        "STIJG_3": stijg_3,
        "STIJG_4": stijg_4,
        "STIJG_5": stijg_5,
        "STIJG_OVERSLAG": stijg_overslag,
        "INDRINGING": indringing,
        "CONSTRAINT_WIDTH": constraint_breedte,
        "CONSTRAINT_START": start_constraint_zone,
    }

## functie creer stix

In [ ]:
class BufferEndingMethod(Enum):
    WITHIN = 1
    PERPENDICULAR = 2
    VERTICAL = 3


@dataclass
class ModelLimits:
    left_limit: float = None
    right_limit: float = None
    lower_limit: float = -50
    upper_limit: float = None


@dataclass
class SoilPolygon:
    soil: SoilStore
    polygon: Polygon
    given_index: int = None
    status: str = None
    aquifer_id: list = field(default_factory=list)

    def y_at_x(self, x: float) -> list[float]:
        # Get the value of the polygon at
        if self.polygon:
            line = LineString([(x, 1e5), (x, -1e5)])
            intersection = self.polygon.intersection(line)
            if intersection.is_empty:
                return []
            if isinstance(intersection, Point):
                return [intersection.xy[1]]
            elif isinstance(intersection, MultiLineString):
                return [v for g in intersection.geoms for v in g.xy[1]]
            elif isinstance(intersection, LineString):
                return list(intersection.xy[1])
        return []


def prevent_overlap_ref(to_correct: ReferenceLine, to_avoid: ReferenceLine, ceil: float = 20) -> ReferenceLine | None:
    if shapely.equals(to_correct.linestring, to_avoid.linestring):
        return None
    if not to_correct.linestring.intersects(to_avoid.linestring):
        return to_correct

    to_correct_ls = shapely.set_precision(to_correct.linestring, 10**-DECIMALS)
    tmp = shapely.difference(to_correct_ls, to_avoid.linestring)
    llim = to_correct.coords[0]
    rlim = to_correct.coords[-1]
    if isinstance(tmp, shapely.geometry.LineString):
        coords = list(tmp.coords)
        nwx = coords[0][0] - 0.01

        while nwx in [c[0] for c in coords]:
            nwx -= 0.001

        coords.insert(0, (nwx, ceil))

        coords.insert(0, (llim[0], ceil))
        nwx = coords[-1][0] + 0.01
        while nwx in [c[0] for c in coords]:
            nwx += 0.001
        coords.append((nwx, ceil))
        coords.append((rlim[0], ceil))
    elif isinstance(tmp, shapely.geometry.MultiLineString):
        return to_correct
    elif isinstance(tmp, shapely.geometry.GeometryCollection):
        return to_correct

    new_x = [c[0] for c in coords]
    new_z = [c[1] for c in coords]
    return ReferenceLine(
        x=new_x,
        z=new_z,
        label=to_correct.label,
        headline_above=to_correct.headline_above,
        headline_below=to_correct.headline_below,
    )


class DStabGeotechModel:
    def __init__(
        self,
        boreholes: list[Borehole],
        soils: list[SoilStore],
        surface: SurfaceLine,
        limits: ModelLimits = ModelLimits(),
    ):
        self.model = DStabilityModel()
        self.model_limits = limits
        self.boreholes = boreholes
        self.surface = surface
        self.soils = soils

        # self.soil_polygons: list[SoilPolygon] = []
        self.staged_soil_polygons: dict[tuple[int, int], SoilPolygon] = {}

    def get_soil_by_code(self, code: str) -> SoilStore:
        for soil in self.soils:
            if soil.code == code:
                return soil
        raise ValueError(f"Soil with code {code} is not present in soils")

    @property
    def coords(self) -> dict[tuple[int, int], list[tuple[float, float]]]:
        return {k: [(x, y) for x, y in zip(*v.exterior.xy)] for k, v in self.exterior.items()}

    @property
    def most_left_upper_point(self) -> dict[tuple[int, int], tuple[float, float]]:
        """Returns the most left upper point of the shape."""
        return {k: sorted(v, key=lambda tup: (tup[0], -tup[1]))[0] for k, v in self.coords.items()}

    @property
    def most_left_lower_point(self) -> dict[tuple[int, int], tuple[float, float]]:
        """Returns the most left lower point of the shape."""
        return {k: sorted(v, key=lambda tup: (tup[0], tup[1]))[0] for k, v in self.coords.items()}

    @property
    def most_right_upper_point(self) -> dict[tuple[int, int], tuple[float, float]]:
        """Returns the most right upper point of the shape."""
        return {k: sorted(v, key=lambda tup: (-tup[0], -tup[1]))[0] for k, v in self.coords.items()}

    @property
    def most_right_lower_point(self) -> dict[tuple[int, int], tuple[float, float]]:
        """Returns the most right lower point of the shape."""
        return {k: sorted(v, key=lambda tup: (-tup[0], tup[1]))[0] for k, v in self.coords.items()}

    @property
    def y_max(self) -> dict[tuple[int, int], float]:
        return {k: sorted(v, key=lambda tup: tup[1])[-1][1] for k, v in self.coords.items()}

    @property
    def y_min(self) -> dict[tuple[int, int], float]:
        return {k: sorted(v, key=lambda tup: tup[1])[0][1] for k, v in self.coords.items()}

    @property
    def x_max(self) -> dict[tuple[int, int], float]:
        return {k: sorted(v, key=lambda tup: tup[0])[-1][0] for k, v in self.coords.items()}

    @property
    def x_min(self) -> dict[tuple[int, int], float]:
        return {k: sorted(v, key=lambda tup: tup[0])[0][0] for k, v in self.coords.items()}

    # def contains(self, point: Point):
    #     if self._shape.buffer(19**-DECIMALS).contains(point):
    #         return True

    @property
    def levee(self) -> dict[tuple[int, int], Polygon]:
        return {
            k: unary_union([sp.polygon for sp in v if sp.status == "LEVEE"])
            for k, v in self.staged_soil_polygons.items()
        }

    @staticmethod
    def centerpoint_between_two_boreholes(borehole1: Borehole, borehole2: Borehole) -> float:
        return (borehole2.x_distance - borehole1.x_distance) / 2 + borehole1.x_distance

    @property
    @cache
    def borehole_x_lims_from_surface(self) -> dict[Borehole, tuple[float, float]]:
        d = {}
        for borehole in self.boreholes:
            but = self.surface.get_cpoint_by_label("buitenteen")
            bukr = self.surface.get_cpoint_by_label("buitenkruin")
            bikr = self.surface.get_cpoint_by_label("binnenkruin")
            bit = self.surface.get_cpoint_by_label("binnenteen")

            if borehole.name.lower() == "buitendijks":
                left = self.model_limits.left_limit if self.model_limits.right_limit else self.surface.left_limit
                right = but.x + (bukr.x - but.x) / 3
            elif borehole.name.lower() == "kruin":
                left = but.x + (bukr.x - but.x) / 3
                right = bit.x - (bit.x - bikr.x) / 3
            elif borehole.name.lower() == "binnendijks":
                left = bit.x - (bit.x - bikr.x) / 3
                right = self.model_limits.right_limit if self.model_limits.right_limit else self.surface.right_limit

            d[borehole] = (left, right)
        return d

    @property
    @cache
    def borehole_x_lims(self) -> dict[Borehole, tuple[float, float]]:
        sorted_boreholes = sorted(self.boreholes, key=lambda b: b.x_distance)
        d = {}
        for idx in range(len(sorted_boreholes)):
            if idx == 0:
                left = self.model_limits.left_limit if self.model_limits.right_limit else self.surface.left_limit
                right = self.centerpoint_between_two_boreholes(sorted_boreholes[idx], sorted_boreholes[idx + 1])
            elif idx == len(sorted_boreholes) - 1:
                left = self.centerpoint_between_two_boreholes(sorted_boreholes[idx - 1], sorted_boreholes[idx])
                right = self.model_limits.right_limit if self.model_limits.right_limit else self.surface.right_limit
            else:
                left = self.centerpoint_between_two_boreholes(sorted_boreholes[idx - 1], sorted_boreholes[idx])
                right = self.centerpoint_between_two_boreholes(sorted_boreholes[idx], sorted_boreholes[idx + 1])

            d[sorted_boreholes[idx]] = (left, right)
        return d

    @property
    def aquifer_ids(self) -> list[int]:
        ids = set()
        for borehole in self.boreholes:
            for layer in borehole.layers:
                for id in layer.aquifer_ids:
                    ids.add(id)
        return list(ids)

    def create_polygons_from_borehole(
        self,
        borehole: Borehole,
    ) -> list[SoilPolygon]:
        polygons = []
        for idx in range(len(borehole.layers)):
            if idx == len(borehole.layers):
                bottom = self.model_limits.lower_limit
            elif idx == len(borehole.layers) - 1:
                bottom = self.model_limits.lower_limit
            else:
                bottom = borehole.layers[idx + 1].top
            top = borehole.layers[idx].top
            left, right = self.borehole_x_lims_from_surface[borehole]
            pointlist = self.create_pointlist(top, bottom, left, right)
            polygon = self.create_polygon(pointlist)
            soil = self.get_soil_by_code(borehole.layers[idx].code)

            soilpolygon = SoilPolygon(polygon=polygon, soil=soil, aquifer_id=borehole.layers[idx].aquifer_ids)

            polygons.append(soilpolygon)

        return polygons

    def set_levee_status_by_soilcode(
        self,  # PVL toegevoegd
        index: tuple[int, int],
        soilcodes: list[str] = ("Dz", "Dk"),
    ):
        for sp in self.staged_soil_polygons[index]:
            if sp.soil.code in soilcodes:
                sp.status = "LEVEE"

    def create_polygons(
        self,
        index: tuple[int, int] = None,
        extend_top_to: float = None,
    ):
        """Creates the combination between borehole layers and soils"""
        if index is None:
            index = (0, 0)

        soil_polygons = []
        for borehole in self.boreholes:
            if extend_top_to is not None:
                borehole.increase_top_of_borehole(extend_top_to)

            polygons = self.create_polygons_from_borehole(borehole)
            soil_polygons.extend(polygons)

        self.staged_soil_polygons[index] = soil_polygons

    @property
    def surface_level(self) -> list[tuple[float, float]]:
        surface_coords = self.surface.clip_coords_between(
            self.surface.coordinates, self.model_limits.left_limit, self.model_limits.right_limit
        )
        return surface_coords

    def add_scenario(self, from_index: int = None) -> tuple[int, int]:
        kys = list(self.staged_soil_polygons.keys())
        last_scenario = kys[-1][0]
        # last_stage = kys[-1][-1]
        if from_index is None:
            from_index = (last_scenario, 0)

        new_index = (last_scenario + 1, 0)
        self.staged_soil_polygons[new_index] = self.staged_soil_polygons[from_index]

        return new_index

    def smash_surface_level(self, index: tuple[int, int] = None, fill_below=True):
        """
        Smash surface level

        Cut the boreholes to the surface level, resulting in the area below
        the surface level. If fill_below, the boreholes will be extended
        upwards such that the full area below the surface line is filled.

        Else gaps may exist between the top of a borehole and the surface area

        :param fill_below: (bool) Whether to fill below surface line
        :return: None
        """
        if index is None:
            index = (0, 0)

        # If fill below surface layer, get the top of the surface layer
        if fill_below:
            max_surface = self.surface.upper_limit + 1  # Add a bit just to be sure...
        else:
            max_surface = None

        # If no soil-polygons or fill below, recalculate the soil-polygons
        if self.staged_soil_polygons[index] == [] or fill_below:
            self.create_polygons(index=index, extend_top_to=max_surface)

        line = LineString(self.surface_level)

        new_soil_polygons = []
        for layer in self.staged_soil_polygons[index]:
            new_polygons = self.split_polygon_by_line(layer.polygon, line, keep_below=True)
            for p in new_polygons:
                new_soil_polygons.append(SoilPolygon(polygon=p, soil=layer.soil, aquifer_id=layer.aquifer_id))

        self.staged_soil_polygons[index] = self.snap_layers(new_soil_polygons)

    @property
    def exterior(self) -> dict[tuple[int, int], Polygon]:
        """
        Polygon of the total model area (union of all soil layers)

        :return: Exterior polygon
        """
        exterior = {k: unary_union([layer.polygon for layer in v]) for k, v in self.staged_soil_polygons.items()}
        return exterior

    def get_y_of_surface_level_at_x(self, index: tuple[int, int], x: float) -> float:
        """
        Gives the y-value of the geometry surface level at a given x value

        :param x: X-coordinate (must be between left and right model limit)
        :return: (float) Y-value at X of the surface line
        """
        return interpolate_coordinates(self.get_surface_line_of_polygon(self.exterior[index]), x)

    def get_x_at_intersection_of_y(self, index: tuple[int, int], y: float) -> list[float]:
        # Gets all the x-coordinates where the surface line intersects the y value
        surface = LineString(self.get_surface_line_of_polygon(self.exterior[index]))
        to_intersect = LineString([(self.x_min[index], y), (self.x_max[index], y)])

        intersections = surface.intersection(to_intersect)
        if intersections.is_empty:
            return []
        if isinstance(intersections, shapely.Point):
            return [intersections.x]
        return sorted([p.x for p in intersections.geoms])

    def get_soil_store_by_code(self, code: str):
        for soil in self.soils:
            if soil.code == code:
                return soil
        raise ValueError(f"No soil with code {code}")

    @staticmethod
    def get_surface_line_of_polygon(polygon: Polygon) -> list[tuple[float, float]]:
        """
        Gets the surface of a polygon, which is the drape down surface

        :param polygon: Input Polygon
        :return surface_coords: List of coordinates
        """
        polygon = ensure_polygon(polygon)  # PvL toegevoegd
        poly_x, poly_y = polygon.exterior.coords.xy
        # lr = LinearRing(coords)
        min_x = min(poly_x)
        max_x = max(poly_x)
        coords_poly = [(x, y) for x, y in zip(poly_x, poly_y)]
        left_points = [c for c in coords_poly if c[0] == min_x]
        right_points = [c for c in coords_poly if c[0] == max_x]

        max_left = max([c[1] for c in left_points])
        max_right = max([c[1] for c in right_points])
        upper_left = (min_x, max_left)
        upper_right = (max_x, max_right)

        index_left = coords_poly.index(upper_left)
        index_right = coords_poly.index(upper_right)

        if index_right > index_left:
            surface_coords = coords_poly[index_left : index_right + 1]
        else:
            tmp = coords_poly[index_left:] + coords_poly[: index_right + 1]
            surface_coords = sorted(tmp, key=lambda t: t[0])
        return surface_coords

    def create_polygon_below_surface(
        self,
        index: tuple[int, int],
        center_x: float,
        soil: SoilStore,
        width: float = 1,
        height: float = 1,
        status: str = "LEVEE",
    ):
        top_y = self.get_y_of_surface_level_at_x(index, center_x)
        bottom_y = top_y - height
        left_x = center_x - width / 2
        right_x = center_x + width / 2

        polygon = Polygon(
            [(left_x, top_y + 10), (right_x, top_y + 10), (right_x, bottom_y), (left_x, bottom_y), (left_x, top_y + 10)]
        )

        splitline = LineString(self.get_surface_line_of_polygon(self.exterior[index]))
        polygons_below = self.split_polygon_by_line(polygon, splitline, keep_below=True)
        if len(polygons_below) != 1:
            raise NotImplementedError("Too many polygons")

        self.smash_polygons(index, [SoilPolygon(polygon=polygons_below[0], soil=soil, status=status)])

    def smash_levee(
        self,
        start_x: float,
        end_x: float,
        soil: SoilStore,
        index: tuple[int, int] = None,
        exclude: list[SoilStore] = None,
        extend_down: bool = True,
    ):  # -> Polygon:
        if index is None:
            index = (0, 0)
        if exclude is None:
            exclude = []

        start_y = self.get_y_of_surface_level_at_x(index, start_x)
        end_y = self.get_y_of_surface_level_at_x(index, end_x)

        delta = (end_y - start_y) / (end_x - start_x)
        increment = 0
        splitline = LineString(
            [(start_x - increment, start_y - increment * delta), (end_x + increment, end_y + increment * delta)]
        )

        lowest_point = min(start_y, end_y)
        if extend_down:
            splitline = LineString([(start_x, start_y), (start_x, lowest_point), (end_x, lowest_point), (end_x, end_y)])

        # Get the polygons that shape the levee
        levee_polygons = self.split_polygon_by_line(self.exterior[index], splitline, keep_below=False)

        included_levee_polygon = []
        for pol in levee_polygons:
            corrected = False
            # Check if any polygon within the levee is part of the exclude list
            for soilpolygon in self.staged_soil_polygons[index]:
                if pol.intersects(soilpolygon.polygon):
                    r = pol.intersection(soilpolygon.polygon)
                    if isinstance(r, shapely.Point):
                        continue
                    if isinstance(r, shapely.LineString):
                        continue
                    if soilpolygon.soil in exclude:  # and r.area > 0.5:
                        # If to exclude, keep the exclude soil-type
                        included_levee_polygon.append((r, soilpolygon.soil, soilpolygon.aquifer_id))
                    else:
                        # Else, assign the levee soil-type to the polygon
                        included_levee_polygon.append((r, soil, soilpolygon.aquifer_id))
                    corrected = True
            if not corrected:
                included_levee_polygon.append((pol, soil))

        levee_polygons_to_add = []
        # Create a list of polygons by unpacking GeometryCollections
        for t in included_levee_polygon:
            if isinstance(t[0], Polygon):
                levee_polygons_to_add.append(t)
            elif isinstance(t[0], MultiPolygon):
                for pol in t[0].geoms:
                    levee_polygons_to_add.append((pol, t[1]))

        # Add each polygon to the polygon collection
        for lp, sl, aq in levee_polygons_to_add:
            self.smash_polygon(polygon=lp, index=index, soil=sl, status="LEVEE", aquifer_id=aq)

            self.staged_soil_polygons[index] = self.snap_layers(self.staged_soil_polygons[index])

    @staticmethod
    def split_polygon_by_line(polygon: Polygon, line: LineString, keep_below: bool = True) -> list[Polygon]:
        to_keep = []
        # set_precision(line, 10 ** -DECIMALS)
        # set_precision(polygon, 10 ** -DECIMALS)
        split_result = split(polygon, line)

        for part in split_result.geoms:
            centerpoint_x = float(part.centroid.xy[0][0])

            y_at_center_x = part.intersection(
                LineString([(centerpoint_x, part.bounds[1] - 1), (centerpoint_x, part.bounds[3] + 1)])
            ).xy[1]

            centerpoint_y = np.mean(y_at_center_x)
            # centerpoint_y = float(part.centroid.xy[1][0])
            x_line, y_line = line.coords.xy
            line_coords = [(round(x, DECIMALS), round(y, DECIMALS)) for x, y in zip(x_line, y_line)]
            y_at_line = interpolate_coordinates(line_coords, centerpoint_x)

            if centerpoint_y < y_at_line:
                # Part is below the line
                if keep_below:
                    to_keep.append(part)
            elif centerpoint_y > y_at_line:
                # Part is above the line
                if not keep_below:
                    to_keep.append(part)

            else:
                raise ValueError("Part is the line")
        return to_keep

    @staticmethod
    def create_pointlist(top: float, bottom: float, left: float, right: float) -> list[tuple[float, float]]:
        top = round(top, DECIMALS)
        bottom = round(bottom, DECIMALS)
        left = round(left, DECIMALS)
        right = round(right, DECIMALS)

        if left >= right:
            raise ValueError("Left boundary must be smaller than right boundary")
        if bottom >= top:
            raise ValueError("Lower boundary must be smaller than top boundary")
        return [(left, top), (right, top), (right, bottom), (left, bottom), (left, top)]

    @staticmethod
    def create_polygon(points: list[tuple]) -> Polygon:
        if len(points) < 3:
            raise ValueError("Too few points, minimum has to be 3")
        if points[0] != points[-1]:
            points.append(points[0])
        return Polygon(points).buffer(0)

    def create_offset_layer(
        self, polygon: Polygon, offset_distance: float, ending_method: BufferEndingMethod = BufferEndingMethod.WITHIN
    ):
        """
        Create an offset line from a list of coordinates with the same number of points.

        Handles edge cases to prevent NaN values.

        Parameters
        ----------
        coordinates : list of tuples
            List of (x, y) coordinate tuples defining the original line
        offset_distance : float
            Distance to offset the line. Positive values offset to the right of the direction of travel,
            negative values offset to the left.

        Returns
        -------
        list of tuples
            List of (x, y) coordinate tuples defining the offset line
        """

        def get_slope(coordinates, idx_a, idx_b):
            x_a = coordinates[idx_a][0]
            x_b = coordinates[idx_b][0]
            if x_a == x_b:
                return 0
            y_a = coordinates[idx_a][1]
            y_b = coordinates[idx_b][1]
            return (y_b - y_a) / (x_b - x_a)

        coordinates = self.get_surface_line_of_polygon(polygon)
        coordinates = list(dict.fromkeys(coordinates))

        def get_zero_intercept(coordinates, idx, slope):
            b = coordinates[idx][1] - coordinates[idx][0] * slope
            return b

        couples = []
        offset_coords = []
        # offset_coords2 = []
        for i in range(len(coordinates)):
            if i == 0:
                slope_a = 0
                slope_b = get_slope(coordinates, i + 1, i)
            elif i == len(coordinates) - 1:
                slope_a = get_slope(coordinates, i, i - 1)
                slope_b = 0
            else:
                slope_a = get_slope(coordinates, i, i - 1)
                slope_b = get_slope(coordinates, i + 1, i)

            angle_a = 0 if slope_a == 0 else math.degrees(math.atan(slope_a))
            angle_b = 0 if slope_b == 0 else math.degrees(math.atan(slope_b))
            offset_dist_b = offset_distance / math.sin(math.radians(90 - angle_a))
            offset_dist_d = offset_distance / math.sin(math.radians(90 - angle_b))
            b = get_zero_intercept(coordinates, i, slope_a) + offset_dist_b
            d = get_zero_intercept(coordinates, i, slope_b) + offset_dist_d
            #
            if i == 0:
                x = coordinates[i][0]
                z = slope_b * coordinates[i][0] + d
            elif i == len(coordinates) - 1:
                x = coordinates[i][0]
                z = slope_a * coordinates[i][0] + b
            else:
                if round(slope_a, 2) == round(slope_b, 2):
                    x = coordinates[i][0]
                else:
                    x = (d - b) / (slope_a - slope_b)
                z = slope_a * x + b

            offset_coords.append((round(x, DECIMALS), round(z, DECIMALS)))
            couples.append(((x, z), (coordinates[i][0], coordinates[i][1])))

        surface_coords = [*coordinates, *offset_coords[::-1]]
        if surface_coords[0] != surface_coords[-1]:
            surface_coords.append(surface_coords[0])

        layer = Polygon(surface_coords)
        layer = layer.buffer(0)
        try:
            layer = set_precision(layer, 10**-DECIMALS)
        except Exception:
            print(layer)
            fig, ax = plt.subplots()
            # plot_polygon(ax, polygon, color='blue', alpha=0.5)
            plot_polygon(ax, layer, color="red", alpha=0.3)
            ax.plot(layer.exterior.coords.xy[0], layer.exterior.coords.xy[1], marker="x", color="black")
            for c in couples:
                ax.plot([c[0][0], c[1][0]], [c[0][1], c[1][1]])

            plt.show()

        if ending_method == BufferEndingMethod.WITHIN:
            buffer_layer = layer.intersection(polygon)
            return buffer_layer
        elif ending_method == BufferEndingMethod.PERPENDICULAR:
            return layer

    @staticmethod
    def create_gl_pointlist(polygon: Polygon) -> list[glPoint]:
        xx, yy = polygon.exterior.coords.xy
        return [glPoint(x=x, z=y) for x, y in zip(xx, yy)]

    @staticmethod
    def add_point_to_polygon(polygon: Polygon, point: tuple[float, float]) -> Polygon:
        new_polygon = add_point_to_polygon(polygon, point)
        return new_polygon

    @staticmethod
    def shapely_to_geolib_point(point: shapely.Point) -> Point:
        return Point(x=round(point.x, DECIMALS), z=round(point.y, DECIMALS))

    def smash_polygons(self, index: tuple[int, int], soilpolygons: list[SoilPolygon]):
        to_smash = [*soilpolygons]
        # staged_soil_polygons = self.staged_soil_polygons[index]
        while len(to_smash) > 0:
            sp = to_smash[0]
            polygon = sp.polygon
            soil = sp.soil
            status = sp.status
            staged_soil_polygons = self.smash_polygon(  # noqa: F841
                index,
                polygon,
                soil,
                status=status,
                # staged_soil_polygons,
                apply_to_self=True,
            )

            to_smash = [s for s in to_smash if s != sp]
            self.staged_soil_polygons[index] = self.snap_layers(self.staged_soil_polygons[index])

    def change_polygon(self, index: tuple[int, int], polygon: Polygon, fun: Callable):
        # apply_to_self = False
        exclude = ["Dk", "Dz", "Z"]
        # exclude = []
        new_soil_polygons = []
        for soillayer in self.staged_soil_polygons[index]:
            existing_polygon = soillayer.polygon
            if existing_polygon.buffer(0).intersects(polygon) and soillayer.soil.code not in exclude:
                overlap = existing_polygon.intersection(polygon).buffer(0)
                remainder = existing_polygon.difference(polygon).buffer(0)
                if isinstance(overlap, MultiPolygon):
                    remainders = list(overlap.geoms)
                else:
                    remainders = [overlap]
                for r in remainders:
                    r = set_precision(r, 10**-DECIMALS)
                    if r.is_empty:  # or r.area <0.01:
                        # The polygon is empty
                        # do not add this one
                        continue
                    new_soil = fun(soillayer.soil)
                    try:
                        self.get_soil_by_code(new_soil.code)
                    except Exception:
                        self.soils.append(new_soil)
                    new_remainder_soilpolygon = SoilPolygon(
                        polygon=r, soil=new_soil, status=soillayer.status, aquifer_id=soillayer.aquifer_id
                    )
                    new_soil_polygons.append(new_remainder_soilpolygon)
                if isinstance(remainder, MultiPolygon):
                    remainders = list(remainder.geoms)
                else:
                    remainders = [remainder]
                for r in remainders:
                    r = set_precision(r, 10**-DECIMALS)
                    if r.is_empty:  # or r.area < 0.1:
                        # The remainder polygon is empty
                        # do not add this one
                        continue
                    new_remainder_soilpolygon = SoilPolygon(
                        polygon=r, soil=soillayer.soil, status=soillayer.status, aquifer_id=soillayer.aquifer_id
                    )
                    new_soil_polygons.append(new_remainder_soilpolygon)
            else:
                # Don't change anything
                new_soil_polygons.append(soillayer)

        self.staged_soil_polygons[index] = new_soil_polygons

        return new_soil_polygons

    @staticmethod
    def add_intersection_points2(polygon_a, polygon_b, tolerance=1e-2):
        # Vind de doorsnede van de grenzen
        boundary_a = polygon_a.boundary
        boundary_b = polygon_b.boundary
        intersection = boundary_a.intersection(boundary_b)

        # Als er geen doorsnede is, probeer dan met een kleine buffer
        if intersection.is_empty:
            buffered_boundary_a = boundary_a.buffer(tolerance)
            buffered_boundary_b = boundary_b.buffer(tolerance)
            intersection = buffered_boundary_a.intersection(buffered_boundary_b)

        # Extract de punten van de doorsnede
        if intersection.geom_type == "Point":
            intersection_points = [intersection]
        elif intersection.geom_type == "MultiPoint" or intersection.geom_type == "GeometryCollection":
            intersection_points = [geom for geom in intersection.geoms if geom.geom_type == "Point"]
        elif intersection.geom_type == "LineString":
            intersection_points = list(intersection.coords)
        else:
            intersection_points = []

        # Voeg de intersectiepunten toe aan polygon_a
        if intersection_points:
            # Converteer polygon_a naar een lijst van coördinaten
            coords_a = list(polygon_a.exterior.coords)

            # Voeg voor elk intersectiepunt het dichtsbijzijnde segment van polygon_a
            new_coords = []
            for i in range(len(coords_a) - 1):
                new_coords.append(coords_a[i])

                # Controleer elk intersectiepunt
                segment = LineString([coords_a[i], coords_a[i + 1]])
                for point in intersection_points:
                    # Als het punt dicht bij dit segment ligt, voeg het toe
                    if isinstance(point, tuple):
                        point_geom = shapely.Point(point)
                    else:
                        point_geom = point

                    if segment.distance(point_geom) < tolerance:
                        if isinstance(point, tuple):
                            new_coords.append(point)
                        else:
                            new_coords.append((point.x, point.y))

            # Voeg het laatste punt toe
            new_coords.append(coords_a[-1])

            # Maak een nieuwe polygon met de aangepaste coördinaten
            updated_polygon_a = Polygon(new_coords)
            return updated_polygon_a

        return polygon_a

    @staticmethod
    def add_intersection_points(polygon1: Polygon, polygon2: Polygon, tolerance=1e-8):
        """
        Add intersection points between two polygons to the first polygon.

        Prevents adding duplicate points that are already in the polygon.

        Args:
            polygon1: The polygon to add points to
            polygon2: The polygon to find intersections with
            tolerance: Distance tolerance for considering points to be duplicates

        Returns
        -------
            A new polygon with intersection points added
        """
        if polygon1 is None or polygon2 is None:
            return polygon1

        if not polygon1.is_valid or not polygon2.is_valid:
            # Fix invalid geometries
            polygon1 = polygon1.buffer(0)
            polygon2 = polygon2.buffer(0)

        # Extract coordinates from the first polygon
        coords = list(polygon1.exterior.coords)

        # Find intersection between the two polygons
        intersection = polygon1.exterior.intersection(polygon2.exterior)

        # No intersections found
        if intersection.is_empty:
            return polygon1

        # Process different types of intersection results
        new_points = []
        if intersection.geom_type == "Point":
            new_points = [intersection]
        elif intersection.geom_type == "MultiPoint":
            new_points = list(intersection.geoms)
        elif intersection.geom_type == "GeometryCollection":
            for geom in intersection.geoms:
                if geom.geom_type == "Point":
                    new_points.append(geom)
                elif geom.geom_type == "LineString":
                    new_points.extend([shapely.Point(p) for p in geom.coords])
        elif intersection.geom_type == "LineString":
            new_points = [shapely.Point(p) for p in intersection.coords]

        # No new points found
        if not new_points:
            return polygon1

        # Filter out duplicate points
        filtered_new_points = []
        for new_point in new_points:
            x, y = new_point.x, new_point.y

            # Check if this point is already in the polygon coordinates
            is_duplicate = False
            for coord in coords:
                if abs(coord[0] - x) < tolerance and abs(coord[1] - y) < tolerance:
                    is_duplicate = True
                    break

            # Only add non-duplicate points
            if not is_duplicate:
                filtered_new_points.append((x, y))

        # If no new unique points, return original polygon
        if not filtered_new_points:
            return polygon1

        # Insert new points into the polygon at appropriate positions
        new_coords = []
        for i in range(len(coords) - 1):  # -1 because last point repeats first
            p1 = coords[i]
            p2 = coords[i + 1]

            # Add the current point
            new_coords.append(p1)

            # Line segment defined by p1 and p2
            line = LineString([p1, p2])

            # Find points that belong on this segment
            segment_points = []
            for point in filtered_new_points:
                point_obj = shapely.Point(point)
                # Check if point is on the line segment
                if line.distance(point_obj) < tolerance:
                    # Calculate distance from p1 to determine order
                    distance = shapely.Point(p1).distance(point_obj)
                    segment_points.append((distance, point))

            # Sort by distance from p1 and add to coordinates
            for _, point in sorted(segment_points):
                new_coords.append(point)

        # Add the last point to close the polygon
        new_coords.append(coords[-1])

        # Create new polygon with updated coordinates
        new_polygon = Polygon(new_coords)

        # Ensure validity
        if not new_polygon.is_valid:
            new_polygon = new_polygon.buffer(0)

        return new_polygon

    @staticmethod
    def snap_points(polygon: Polygon, soil_polygons: list[SoilPolygon], proximity_threshold=0.01) -> Polygon:
        # Snap new polygon points to existing points if they are too close
        snapped_points = []
        c_points = []
        for point in polygon.exterior.coords:
            snapped = False
            for poly in soil_polygons:
                for existing_point in poly.polygon.exterior.coords:
                    if shapely.Point(point).distance(shapely.Point(existing_point)) < proximity_threshold:
                        snapped_points.append(existing_point)
                        c_points.append(existing_point)
                        snapped = True
                        break
                    # elif 0 < shapely.Point(existing_point).distance(polygon) < proximity_threshold:
                    #
                    #     print(f"This point needs to be snapped too: {existing_point}")
                    #     if existing_point not in snapped_points:
                    #         snapped_points.append(existing_point)
                if snapped:
                    break
            if not snapped:
                snapped_points.append(point)

        snapped_polygon = Polygon(snapped_points)

        if len(snapped_polygon.exterior.coords) != len(polygon.exterior.coords):
            print("mismatch")

        return snapped_polygon

    def smash_polygon(
        self,
        index: tuple[int, int],
        polygon: Polygon,
        soil: SoilStore,
        status: str = None,
        aquifer_id: list[int] = None,
        apply_to_self: bool = True,
    ) -> Polygon:
        """
        Adds a polygon to the geometry and removes all area from existing polygons.

        :param polygon: Polygon to be added.
        :param soil: SoilStore object associated with the polygon.
        :param soil_polygons: List of existing SoilPolygon objects.
        :param status: Status of the new polygon.
        :param min_area: Minimum area threshold for polygons.
        :param proximity_threshold: Distance threshold for snapping points.
        :return: List of updated SoilPolygon objects.
        """
        soil_polygons = self.staged_soil_polygons[index]

        new_soil_polygons = []

        # simplified_polygon = set_precision(simplified_polygon, 10 ** - DECIMALS)
        snapped_polygon = Polygon(polygon.exterior.coords).buffer(0)
        snapped_polygon = set_precision(snapped_polygon, 10**-DECIMALS)
        snapped_polygon = self.snap_points(snapped_polygon, soil_polygons)
        snapped_polygon = set_precision(snapped_polygon.buffer(0), 10**-DECIMALS)
        # snapped_polygon = set_precision(snapped_polygon, 10**-DECIMALS)

        # fig, ax = plt.subplots()
        # plot_polygon(ax, snapped_polygon)
        # ax.scatter(snapped_polygon.exterior.coords.xy[0],
        #            snapped_polygon.exterior.coords.xy[1],
        #            color='black')
        # plt.show()

        for soillayer in soil_polygons:
            existing_polygon = soillayer.polygon

            if existing_polygon.intersects(snapped_polygon):
                existing_polygon = set_precision(existing_polygon, 10**-DECIMALS)
                snapped_polygon = self.add_intersection_points(snapped_polygon, existing_polygon)
                snapped_polygon = set_precision(snapped_polygon.buffer(0), 10**-DECIMALS)  #

                # fig, ax = plt.subplots()
                # plot_polygon(ax, existing_polygon, color='red', alpha=0.5)
                # plot_polygon(ax, snapped_polygon, color='blue', alpha=0.5)
                # ax.scatter(existing_polygon.exterior.xy[0],
                #            existing_polygon.exterior.xy[1],
                #            marker='x')
                # ax.scatter(snapped_polygon.exterior.xy[0],
                #            snapped_polygon.exterior.xy[1],
                #            marker='o')
                # x = [p[0] for p in snapped_polygon.exterior.coords]
                # z = [p[1] for p in snapped_polygon.exterior.coords]
                # ax.plot(x, z)
                # for idx, (xx, zz) in enumerate(zip(x, z)):
                #     # ax.scatter(xx,zz)
                #     ax.annotate(str(idx), (xx, zz))
                # plt.show()

                remainder = existing_polygon.difference(snapped_polygon)
                if isinstance(remainder, MultiPolygon) or isinstance(remainder, GeometryCollection):
                    remainders = list(remainder.geoms)
                else:
                    remainders = [remainder]
                for r in remainders:
                    if r.is_empty or isinstance(r, LineString):
                        continue
                    new_remainder_soilpolygon = SoilPolygon(
                        polygon=r, soil=soillayer.soil, status=soillayer.status, aquifer_id=soillayer.aquifer_id
                    )
                    new_soil_polygons.append(new_remainder_soilpolygon)
            else:
                new_soil_polygons.append(soillayer)

        if aquifer_id is None:
            aquifer_id = []

        if not snapped_polygon.is_empty:
            new_soilpolygon = SoilPolygon(polygon=snapped_polygon, soil=soil, status=status, aquifer_id=aquifer_id)
            new_soil_polygons.append(new_soilpolygon)

        if apply_to_self:
            self.staged_soil_polygons[index] = new_soil_polygons

        return snapped_polygon

    @staticmethod
    def snap_layers(soil_polygons: list[SoilPolygon]):
        """
        Ensure that all points are in touching layers by the Shapely Snap method

        :return:
        """
        to_snap = True
        n = 0

        sorted_soil_polygons = sorted(soil_polygons, key=lambda p: len(p.polygon.exterior.coords), reverse=True)
        to_loop = deque(sorted_soil_polygons)
        while to_snap or n < len(to_loop) + 1:
            to_snap = False
            total_polygon = None
            for soil_polygon in to_loop:
                before = soil_polygon.polygon
                if not total_polygon:
                    total_polygon = soil_polygon.polygon
                    continue
                snapped_polygon = snap(soil_polygon.polygon, total_polygon, tolerance=10 ** -(DECIMALS + 1))
                snapped_polygon = set_precision(snapped_polygon, 10**-DECIMALS)
                if snapped_polygon.is_empty:
                    continue
                soil_polygon.polygon = snapped_polygon
                try:
                    equal = soil_polygon.polygon.equals(before)
                except Exception:
                    equal = True
                if not equal:
                    to_snap = True
                total_polygon = unary_union([soil_polygon.polygon, total_polygon])

            # touched = self.ensure_shared_boundaries([s.polygon for s in sorted_soil_polygons])
            to_loop.rotate(-1)
            n += 1

        return sorted_soil_polygons

    @staticmethod
    def ensure_shared_boundaries(polygons: list[Polygon]):
        # Step 1: Convert all polygons to a single multilinestring (the boundaries)
        boundaries = [polygon.boundary for polygon in polygons]

        # Step 2: Merge all boundaries into a single geometry
        all_boundaries = unary_union(boundaries)

        # Step 3: Reconstruct polygons from the merged boundaries
        # This ensures vertices are shared exactly
        new_polygons = list(polygonize(all_boundaries))

        return new_polygons

    def show_model(
        self,
        index: tuple[int, int],
    ):
        _, ax = plt.subplots()
        labels = []
        for layer in self.staged_soil_polygons[index]:
            plot_polygon(ax, layer.polygon, label=layer.soil.name, facecolor=layer.soil.color[0:7], alpha=0.5)
            ax.scatter(layer.polygon.exterior.xy[0], layer.polygon.exterior.xy[1], marker="x", color="black")

            proxy = Patch(facecolor=layer.soil.color[0:7], edgecolor="black", alpha=0.5, label=layer.soil.name)
            labels.append(proxy)

        ax.legend(handles=labels)

        plt.show()

    def add_headline(self, index: tuple[int, int], headline: HeadLine, scenario_id: int, stage_id: int) -> int:
        headline.cut_to_lims(self.x_min[index], self.x_max[index])

        given_id = self.model.add_head_line(
            headline.geolib_pointlist,
            scenario_index=scenario_id,
            stage_index=stage_id,
            label=headline.label,
            notes=headline.notes,
            is_phreatic_line=headline.is_phreatic,
        )
        headline.given_id = given_id
        return given_id

    def add_referenceline(self, index: tuple[int, int], referenceline: ReferenceLine, scenario_id: int, stage_id: int):
        referenceline.cut_to_lims(self.x_min[index], self.x_max[index])

        self.model.add_reference_line(
            referenceline.geolib_pointlist,
            scenario_index=scenario_id,
            stage_index=stage_id,
            label=referenceline.label,
            notes=referenceline.notes,
            top_head_line_id=None if referenceline.headline_above is None else referenceline.headline_above.given_id,
            bottom_headline_id=None if referenceline.headline_below is None else referenceline.headline_below.given_id,
        )

    def get_soillayers_at_x(self, index: tuple[int, int], x: float, soillayers: list[SoilPolygon]) -> list[SoilPolygon]:
        vertical = LineString([(x, self.y_max[index]), (x, self.y_min[index])])
        found_soillayers = []
        for soillayer in soillayers:
            if soillayer.polygon.buffer(0).intersects(vertical):
                right_boundary = max(soillayer.polygon.exterior.coords.xy[0])
                if x == right_boundary and x != self.x_max[index]:
                    continue
                found_soillayers.append(soillayer)
        return found_soillayers

    def get_boundary_of_layer_at_x(
        self, index: tuple[int, int], soillayer: SoilPolygon, x: float
    ) -> tuple[float, float]:
        vertical = LineString([(x, self.y_max[index]), (x, self.y_min[index])])
        intersections = soillayer.polygon.intersection(vertical)
        if isinstance(intersections, LineString):
            levels = [c[1] for c in intersections.coords]
        elif isinstance(intersections, MultiLineString):
            levels = [c[1] for g in intersections.geoms for c in g.coords]
        elif isinstance(intersections, shapely.Point):
            levels = [intersections.y]
        else:
            raise TypeError("Unexpected geometry type")
        return max(levels), min(levels)

    def evaluate_stresses(
        self, index: tuple[int, int], x: float, y: float, headline: HeadLine, soillayers: list[SoilPolygon]
    ):
        phreatic_level = headline.value_at_x(x)
        sigma = 0
        surface_level = self.get_y_of_surface_level_at_x(index, x)
        sigma += max(0, phreatic_level - surface_level) * 9.81
        for soillayer in self.get_soillayers_at_x(index, x, soillayers):
            top, bottom = self.get_boundary_of_layer_at_x(index, soillayer, x)
            top = max(top, y)
            bottom = max(bottom, y)
            layer_unsat = max(0, top - max(bottom, phreatic_level))
            layer_sat = max(0, min(top, phreatic_level) - bottom)
            sigma += layer_sat * soillayer.soil.saturated_weight.mean
            sigma += layer_unsat * soillayer.soil.unsaturated_weight.mean

        return sigma

    def calculate_uplift_factor(
        self, index: tuple[int, int], x: float, y: float, phreatic_line: HeadLine, headline: HeadLine, soillayers
    ) -> tuple[float, float, float]:
        stress = self.evaluate_stresses(index, x, y, phreatic_line, soillayers)
        head = headline.value_at_x(x)

        water_pressure = (head - y) * 9.81
        uplift_factor = stress / water_pressure
        return uplift_factor, stress, water_pressure

    def create_uplift_area(self, index: tuple[int, int], x_left: float, x_right: float, refline: ReferenceLine):
        polygon = Polygon(
            [
                (x_left, self.y_min[index]),
                (x_right, self.y_min[index]),
                (x_right, self.y_max[index]),
                (x_left, self.y_max[index]),
                (x_left, self.y_min[index]),
            ]
        )
        uplift_polygons = []
        for soilpolygon in self.staged_soil_polygons[index]:
            y_bottom = refline.value_at_x(soilpolygon.polygon.centroid.x)
            if soilpolygon.polygon.centroid.y <= y_bottom:
                continue
            uplift_polygon = polygon.intersection(soilpolygon.polygon)
            if uplift_polygon.is_empty or uplift_polygon.area < 0.01:
                continue
            if not isinstance(uplift_polygon, shapely.Polygon):
                continue
            uplift_soil = soilpolygon.soil.zero_strength_model()

            if uplift_soil not in self.soils:
                self.soils.append(uplift_soil)
                self.model.add_soil(uplift_soil.geolib_soil)

            uplift_polygons.append(
                SoilPolygon(
                    polygon=uplift_polygon,
                    soil=uplift_soil,
                    status=soilpolygon.status,
                    aquifer_id=soilpolygon.aquifer_id,
                )
            )

        self.smash_polygons(index, uplift_polygons)

    def thickness_at_x(self, index: tuple[int, int], x: float, y_bottom: float):
        top = None
        bottom = None
        stage_soil_polygons = self.staged_soil_polygons[index]
        for soillayer in self.get_soillayers_at_x(index, x, stage_soil_polygons):
            if top is None:
                top = self.get_boundary_of_layer_at_x(index, soillayer, x)[0]
            else:
                top = max(top, self.get_boundary_of_layer_at_x(index, soillayer, x)[0])
            if bottom is None:
                bottom = self.get_boundary_of_layer_at_x(index, soillayer, x)[1]
            else:
                bottom = min(bottom, self.get_boundary_of_layer_at_x(index, soillayer, x)[1])

        thickness = top - max(bottom, y_bottom)

        return thickness

    def ditch_check(self, bottom: float):
        labels = ["start sloot", "eind sloot", "start bodem sloot", "eind bodem sloot"]
        cpoint_labels = [p.label for p in self.surface.characteristic_points]
        mask = [label in cpoint_labels for label in labels]
        if not all(mask):
            return None

        start = self.surface.get_cpoint_by_label("start sloot")
        end = self.surface.get_cpoint_by_label("eind sloot")
        start_bottom = self.surface.get_cpoint_by_label("start bodem sloot")
        end_bottom = self.surface.get_cpoint_by_label("eind bodem sloot")

        B = end.x - start.x
        b = end_bottom.x - start_bottom.x

        h1 = min(start.z, end.z) - bottom
        h2 = min(start_bottom.z, end_bottom.z) - bottom

        center_x = (end_bottom.x - start_bottom.x) / 2 + start_bottom.x
        # center_y = self.get_y_of_surface_level_at_x(center_x)

        surface = self.get_surface_line_of_polygon(self.exterior)
        ditch = [c for c in surface if start.x <= c[0] <= end.x]
        ditch_line = LineString(ditch)
        line1 = LineString([(center_x - 100, bottom + 200), (center_x, bottom)])
        line2 = LineString([(center_x + 100, bottom + 200), (center_x, bottom)])
        inter1 = line1.intersection(ditch_line)
        inter2 = line2.intersection(ditch_line)

        if isinstance(inter1, shapely.Point):
            intersect1 = inter1.y
        elif isinstance(inter1, GeometryCollection):
            intersect1 = min([p.y for p in inter1.geoms])
        else:
            intersect1 = None
        if isinstance(inter2, shapely.Point):
            intersect2 = inter2.y
        elif isinstance(inter2, GeometryCollection):
            intersect2 = min([p.y for p in inter2.geoms])
        else:
            intersect2 = None

        intersect = min([intersect1, intersect2], default=None)
        if intersect is not None:
            h3 = intersect - bottom
        else:
            h3 = None

        if B > h1:
            return None
        if b < h2:
            return h2
        return h3

    # def check_for_uplift(
    #     self,
    #     index: tuple[int, int],
    #     stage_config: Stage,
    #     high_head: HeadLine = None,
    #     high_reference: ReferenceLine = None,
    # ) -> HeadLine | None:
    #     phreatic_line = stage_config.phreatic_line
    #     uplift_settings = stage_config.uplift

    #     if high_head is None:
    #         # When no headline assigned no uplift check can can be performed
    #         return None
    #     if uplift_settings.start_x is None:
    #         start_x = self.x_min[index]
    #     else:
    #         start_x = uplift_settings.start_x
    #     if uplift_settings.end_x is None:
    #         # Set the end of the model as x-end when no value is defined
    #         x_end = self.x_max[index]
    #     else:
    #         x_end = uplift_settings.end_x

    #     # Get all soil_polygon in this scenario/stage combination
    #     stage_soil_polygons = self.staged_soil_polygons[index]

    #     # Get all unique x-values in the model
    #     all_x_vals = []
    #     for soillayer in stage_soil_polygons:
    #         all_x_vals = [*all_x_vals, *[c[0] for c in soillayer.polygon.exterior.coords]]
    #     all_x = set(all_x_vals)
    #     all_x = sorted(all_x)
    #     all_x = [x for x in all_x if start_x <= x <= x_end]

    #     # Check the uplift-factor al all x-values
    #     ulf = []
    #     # all_x = np.arange(start_x,x_end,0.05)
    #     for x in all_x:
    #         # print(f"Checking for uplift at x={x}")
    #         y = high_reference.value_at_x(x)
    #         # self.ditch_check(y)
    #         uplift_factor, stress, water_pressure = self.calculate_uplift_factor(
    #             index, x, y, phreatic_line, high_head, stage_soil_polygons
    #         )
    #         ulf.append((x, uplift_factor, stress, water_pressure))

    #     sorted_ulf = sorted(ulf, key=lambda tup: tup[1])

    #     # Apply the uplift zone at the first x-value when criteria is met

    #     if len(sorted_ulf) > 0:
    #         ulf = sorted_ulf[0][1]
    #         ulf_new = sorted_ulf[0][1]
    #         x_coord = sorted_ulf[0][0]
    #         wp = sorted_ulf[0][3]
    #         strss = sorted_ulf[0][2]
    #         x_new = sorted_ulf[0][0]

    #         if ulf <= uplift_settings.safety_factor:
    #             while ulf_new <= ulf:
    #                 x_new -= 0.05
    #                 y = high_reference.value_at_x(x_new)
    #                 ulf_new, stress, water_pressure = self.calculate_uplift_factor(
    #                     index, x_new, y, phreatic_line, high_head, stage_soil_polygons
    #                 )

    #                 print(f"Checked for uplift at {x_new}, SF = {ulf_new}, start was {ulf}")

    #                 if ulf_new < ulf:
    #                     x_coord = x_new
    #                     # wp = water_pressure
    #                     ulf = ulf_new
    #                     strss = stress

    #             print(f"Uplift at {x_coord} is {ulf} ")
    #             thickness = self.thickness_at_x(
    #                 index, round(x_coord, DECIMALS), y_bottom=high_reference.value_at_x(round(x_coord, DECIMALS))
    #             )

    #             # Apply uplift zone
    #             width = thickness * uplift_settings.width_ratio
    #             left_x = x_coord  # - width/2
    #             right_x = x_coord + width
    #             if thickness <= uplift_settings.max_layer_thickness:
    #                 self.create_uplift_area(index, left_x, right_x, high_reference)
    #             if ulf <= 1:
    #                 y = high_reference.value_at_x(x_coord)
    #                 grenspotentiaal = strss / 9.81 + y
    #                 new_headline = headline_uplift(index, self, high_head, grenspotentiaal, left_x, right_x)

    #                 self.add_headline(index, new_headline, index[0], index[1])

    #                 return new_headline
    #             return None
    #         return None
    #     return None

    def get_index_of_calculationsettings_by_id(self, id: str) -> int:
        for index, c in enumerate(self.model.datastructure.calculationsettings):
            if c.Id == id:
                return index
        raise IndexError(f"No CalculationSettings with id {id}")

    def get_scenario_id_by_index(self, index: int) -> int:
        return int(self.model.datastructure.scenarios[index].Id)

    def get_stage_id_by_index(self, scenario_index: int, stage_index: int) -> int:
        return int(self.model.datastructure.scenarios[scenario_index].Stages[stage_index].Id)

    def get_stage(self, scenario_id: int, stage_id: int) -> Stage:
        try:
            return self.model.datastructure.scenarios[scenario_id].Stages[stage_id]
        except Exception:
            raise ValueError("No stage and or scenario with given index")

    def get_list_index_by_attr_id(self, id: str, attr: str):
        if not hasattr(self.model.datastructure, attr):
            raise ValueError(f"No attribute with name {attr}")
        for index, c in enumerate(getattr(self.model.datastructure, attr)):
            if c.Id == id:
                return index
        raise IndexError(f"Attribute {attr} has no item with id {id}")

    def add_state_correlations(self, method: str, scenario_id: int, stage_id: int):
        if method not in ["all", "within_borehole"]:
            raise ValueError("Method type not allowed")

        state_index = self.get_list_index_by_attr_id(
            self.model.datastructure.scenarios[scenario_id].Stages[stage_id].StateId, "states"
        )

        stage_states = self.model.datastructure.states[state_index]

        if method == "all":
            states = []
            for state in stage_states.StatePoints:
                states.append(state.Id)
                self.model.add_state_correlation(
                    correlated_state_ids=states, scenario_index=scenario_id, stage_index=stage_id
                )
        elif method == "within_borehole":
            for borehole, lims in self.borehole_x_lims_from_surface.items():
                states = []

                geometry_index = self.get_list_index_by_attr_id(
                    self.get_stage(scenario_id, stage_id).GeometryId, "geometries"
                )
                layers = self.model.datastructure.geometries[geometry_index].Layers
                lyrs_map = {layer.Id: layer for layer in layers}
                for state in stage_states.StatePoints:
                    layer = lyrs_map[state.LayerId]
                    min_x = min([p.X for p in layer.Points])
                    max_x = max([p.X for p in layer.Points])
                    if min_x >= lims[0] and max_x <= lims[1]:
                        states.append(state.Id)

                self.model.add_state_correlation(
                    correlated_state_ids=states, scenario_index=scenario_id, stage_index=stage_id
                )

    def set_soil_visualisation(self):
        for soil in self.model.soils.Soils:
            soilstore = self.get_soil_store_by_code(soil.Code)
            color = soilstore.color
            shading = PSE[soilstore.dash]
            visual = PersistableSoilVisualization(Color=color, PersistableShadingType=shading, SoilId=soil.Id)
            self.model.datastructure.soilvisualizations.SoilVisualizations.append(visual)

    def create_model(self, model_config: list, savename: str):
        soils_index = {}

        # Remove all default soils in the model
        self.model.datastructure.soils.Soils = []

        # Add all soils
        for soil in self.soils:
            # Add all soils to the model
            given_index = self.model.add_soil(soil.geolib_soil)
            # Keep track of the given layer_ids
            soils_index[soil] = given_index

        for scenario_index, scenario_config in enumerate(model_config):
            label = scenario_config.label
            note = scenario_config.note

            if not self.model.datastructure.has_scenario(scenario_index):
                scenario_id = self.model.add_scenario(label, note)
            else:
                scenario_id = self.model.get_scenario_index(None)
                self.model.datastructure.scenarios[scenario_id].Label = label
                self.model.datastructure.scenarios[scenario_id].Notes = note

            ###
            # Add calculations
            ###
            for c, calculation in enumerate(scenario_config.calculations):
                label = calculation.label
                note = calculation.note
                if not self.model.datastructure.has_calculation(scenario_index, c):
                    calculation_id = self.model.add_calculation(scenario_id, label, note)

                else:
                    calculation_id = self.model.get_calculation_index(None)
                    self.model.datastructure.scenarios[scenario_id].Calculations[calculation_id].Label = label
                    self.model.datastructure.scenarios[scenario_id].Calculations[calculation_id].Notes = note

                calc_settings_id = (
                    self.model.datastructure.scenarios[scenario_id].Calculations[calculation_id].CalculationSettingsId
                )
                settings_index = self.get_list_index_by_attr_id(calc_settings_id, "calculationsettings")

                if calculation.analysismethod:
                    self.model.set_model(
                        calculation.analysismethod, scenario_index=scenario_id, calculation_index=calculation_id
                    )

                self.model.datastructure.calculationsettings[
                    settings_index
                ].CalculationType = calculation.calculationtype
                self.model.datastructure.calculationsettings[settings_index].ModelFactorMean = 1.005
                self.model.datastructure.calculationsettings[settings_index].ModelFactorStandardDeviation = 0.033

            for stage_index, stage_config in enumerate(scenario_config.stages):
                label = stage_config.label
                note = stage_config.note
                if (scenario_index, stage_index) not in self.staged_soil_polygons:
                    self.staged_soil_polygons[(scenario_index, stage_index)] = self.staged_soil_polygons[
                        (scenario_index, 0)
                    ]
                if not self.model.datastructure.has_stage(scenario_index, stage_index):
                    stage_id = self.model.add_stage(scenario_id, label, note)

                else:
                    stage_id = self.model.get_stage_index(None)
                    self.model.datastructure.scenarios[scenario_id].Stages[stage_id].Label = label
                    self.model.datastructure.scenarios[scenario_id].Stages[stage_id].Notes = note

                for headline in stage_config.headlines:
                    self.add_headline((scenario_index, stage_index), headline, scenario_id, stage_id)

                # rl_head = None
                # for referenceline in stage_config.referencelines:
                #     if referenceline.refs_phreatic:
                #         rl_head = referenceline

                # Check for uplift
                # for referenceline in stage_config.referencelines:
                #     if referenceline.has_head:
                #         if referenceline.headline_above.head and stage_config.uplift.check:
                #             print(
                #                 f"Checking for uplift in {scenario_index} and stage {stage_id} for rf {referenceline.label}"
                #             )
                #             uplift_headline = self.check_for_uplift(
                #                 (scenario_index, stage_index),
                #                 stage_config,
                #                 referenceline.headline_above,
                #                 referenceline,
                #             )
                #             if uplift_headline:
                #                 referenceline.headline_above = uplift_headline

                #     if referenceline == rl_head or rl_head is None:
                #         corrected_refline = referenceline
                #     else:
                #         corrected_refline = prevent_overlap_ref(referenceline, rl_head)
                #     if corrected_refline:
                #         self.add_referenceline(
                #             index=(scenario_index, stage_index),
                #             referenceline=corrected_refline,
                #             scenario_id=scenario_id,
                #             stage_id=stage_id,
                #         )

                for idx, layer in enumerate(self.staged_soil_polygons[(scenario_index, stage_index)]):
                    # Create a list of geolib points
                    points = self.create_gl_pointlist(layer.polygon)

                    given_index = self.model.add_layer(
                        points,
                        layer.soil.code,
                        label=f"L{idx}",
                        notes=f"{scenario_index}-{stage_index} - {layer.aquifer_id}",
                        scenario_index=scenario_id,
                        stage_index=stage_id,
                    )

                    layer.given_index = given_index

                    if stage_config.add_state:
                        given_index = self.model.add_state_point(
                            DStabilityStatePoint(
                                layer_id=layer.given_index,
                                point=self.shapely_to_geolib_point(layer.polygon.representative_point()),
                                stress=layer.soil.to_dstability_stress("pop"),
                                is_probabilistic=layer.soil.pop.is_probabilistic,
                            ),
                            scenario_id,
                            stage_id,
                        )

                if stage_config.correlate_state is not None:
                    self.add_state_correlations(stage_config.correlate_state, scenario_id, stage_id)

        self.model.datastructure.projectinfo.Remarks = "Automatisch gegenereerd"
        self.model.datastructure.projectinfo.Project = "LBO2 16-1 en 16-2"

        self.set_soil_visualisation()

        self.model.serialize(rf"{uitvoer}\{savename}.stix")

        print("""
        Created the .stix file
        """)


def set_mohr_coulomb(soil: SoilStore) -> SoilStore:
    # print(soil.shear_strength_above_phreatic_line)
    if (
        soil.shear_strength_above_phreatic_line == ShearStrengthModel.MOHR_COULOMB_CLASSIC
        and soil.shear_strength_below_phreatic_line == ShearStrengthModel.MOHR_COULOMB_CLASSIC
    ):
        return soil
    return soil.mohr_coulomb_model()


def ensure_polygon(geom):
    if geom is None or geom.is_empty:
        raise ValueError("Lege geometrie (None/is_empty) waar Polygon verwacht werd.")

    if isinstance(geom, Polygon):
        return geom

    if isinstance(geom, MultiPolygon):
        # kies de grootste polygon (meestal wat je wil)
        return max(geom.geoms, key=lambda g: g.area)

    if isinstance(geom, GeometryCollection):
        polys = [g for g in geom.geoms if isinstance(g, Polygon) and not g.is_empty]
        if not polys:
            # soms zitten er MultiPolygons in de collectie
            mpolys = [g for g in geom.geoms if isinstance(g, MultiPolygon) and not g.is_empty]
            if mpolys:
                mp = max(mpolys, key=lambda g: g.area)
                return max(mp.geoms, key=lambda g: g.area)
            raise ValueError(f"GeometryCollection zonder Polygon onderdelen: {[g.geom_type for g in geom.geoms]}")
        return max(polys, key=lambda g: g.area)

    raise TypeError(f"Onverwacht type: {type(geom)} ({getattr(geom, 'geom_type', 'unknown')})")

## Functie creeeren stix

In [ ]:
def create_file(path_factsheet, savename):
    # template inladen
    data_from_excel = parse_excel_template(path_factsheet)
    soils = parse_from_excel(path_materials)
    # geometrie inladen
    surface = data_from_excel["SurfaceLine"]
    boreholes = data_from_excel["Boreholes"]
    DL_frea = data_from_excel["DL_FREA"]
    DL_stijg = data_from_excel["DL_STIJG"]
    RL_frea = data_from_excel["RL_FREA"]
    RL_stijg = data_from_excel["RL_STIJG"]
    RL_indringing = data_from_excel["RL_INDRINGING"]
    RL_overslag = data_from_excel["RL_OVERSLAG"]
    constraint_width = data_from_excel["CONSTRAINT_WIDTH"]
    constraint_start = data_from_excel["CONSTRAINT_START"]
    indringing = data_from_excel["INDRINGING"]

    RL_frea.headline_above = DL_frea
    RL_stijg.headline_above = DL_stijg
    RL_indringing.headline_above = DL_stijg
    for soil in soils:
        # Zet alle volgende parameters op probabilistisch
        soil.set_probabilistic(["friction_angle", "undrained_shear_strength_ratio", "pop"])
    # Versimpel het profiel om het aantal punten te verminderen
    surface.simplify()

    # Creer het D-Stab model
    model = DStabGeotechModel(
        boreholes=boreholes,
        soils=soils,
        surface=surface,
        limits=ModelLimits(left_limit=-100, right_limit=100, lower_limit=-20),
    )

    bit = model.surface.get_cpoint_by_label("binnenteen")
    but = model.surface.get_cpoint_by_label("buitenteen")
    bukr = model.surface.get_cpoint_by_label("buitenkruin")
    bikr = model.surface.get_cpoint_by_label("binnenkruin")

    mid_bitalud = (bit.x - bikr.x) / 2 + bikr.x
    # First create a rectangular polygon model from the boreholes
    model.create_polygons()

    # Then apply the surface level, where we fill-out all areas below
    model.smash_surface_level(fill_below=True)

    model_config = []

    # Dagelijkse situatie rekenstage toevoegen
    scenario_index = 0

    cnstr = DStabilitySlipPlaneConstraints(
        is_zone_a_constraints_enabled=True,
        width_zone_a=constraint_width[0],
        x_left_zone_a=constraint_start[0],
        is_zone_b_constraints_enabled=True,
        width_zone_b=50,
        x_left_zone_b=bikr.x,
    )
    cnstr_shallow = DStabilitySlipPlaneConstraints(
        is_zone_a_constraints_enabled=True,
        width_zone_a=bikr.x - but.x,
        x_left_zone_a=but.x,
        is_zone_b_constraints_enabled=True,
        width_zone_b=bit.x - bikr.x + 10,
        x_left_zone_b=bikr.x,
    )

    default_particle_swarm_shallow = DStabilityUpliftVanParticleSwarmAnalysisMethod(
        options_type="Thorough",
        search_area_a=DStabilitySearchArea(height=20.0, top_left=Point(x=bikr.x, z=bikr.z + 20), width=30.0),
        search_area_b=DStabilitySearchArea(
            height=20.0, top_left=Point(x=mid_bitalud, z=bikr.z), width=(bit.x - mid_bitalud) + 5
        ),
        tangent_area_height=15,
        tangent_area_top_z=bikr.z - 1,
        slip_plane_constraints=cnstr_shallow,
    )
    default_particle_swarm_deep = DStabilityUpliftVanParticleSwarmAnalysisMethod(
        options_type="Thorough",
        search_area_a=DStabilitySearchArea(height=20.0, top_left=Point(x=bukr.x, z=bukr.z + 20), width=30.0),
        search_area_b=DStabilitySearchArea(height=20.0, top_left=Point(x=mid_bitalud, z=bikr.z + 10), width=30),
        tangent_area_height=15,
        tangent_area_top_z=bikr.z - 1,
        slip_plane_constraints=cnstr,
    )

    default_prob = DStabilityUpliftVanAnalysisMethod(
        first_circle=DStabilityCircle(
            center=Point(x=bikr.x, z=bikr.z),
            radius=10.0,
        ),
        second_circle_center=Point(x=bit.x, z=bit.z),
    )

    calculations = [
        Calculation(
            label="Semi-prob (met reststerkte)",
            analysismethod=default_particle_swarm_shallow,
            calculationtype=CalculationTypeEnum.DESIGN,
        ),
        Calculation(
            label="Semi-prob (zonder reststerkte)",
            analysismethod=default_particle_swarm_deep,
            calculationtype=CalculationTypeEnum.DESIGN,
        ),
        Calculation(
            label="Probabilistisch (met reststerkte) (FORM)",
            analysismethod=default_prob,
            calculationtype=CalculationTypeEnum.PROBABILISTIC,
        ),
        Calculation(
            label="Probabilistisch (zonder reststerkte) (FORM)",
            analysismethod=default_prob,
            calculationtype=CalculationTypeEnum.PROBABILISTIC,
        ),
    ]

    stage_dagelijks = Stage(
        label="Dagelijks conditie",
        headlines=[DL_frea, DL_stijg],
        referencelines=[RL_frea, RL_stijg],
        add_state=True,
        correlate_state="within_borehole",
        uplift=UpliftSettings(check=False),
    )

    model_config.append(
        Scenario(label="Dagelijks", stages=[stage_dagelijks], calculations=calculations, index=scenario_index)
    )
    model.smash_levee(
        model.surface.get_cpoint_by_label("buitenteen").x,
        model.surface.get_cpoint_by_label("binnenteen").x,
        model.get_soil_by_code("Dk"),
        (scenario_index, 0),
        exclude=[model.get_soil_by_code("T"), model.get_soil_by_code("Dz")],
    )
    # Verander gedraineerd grondgedrag in  bovenste 0.8 m
    surface_buffer_top = model.create_offset_layer(model.exterior[(scenario_index, 0)], offset_distance=-0.8)
    model.change_polygon((scenario_index, 0), surface_buffer_top, set_mohr_coulomb)

    # voeg een toplaag toe

    model.set_levee_status_by_soilcode((scenario_index, 0))

    levee_buffer_top = model.create_offset_layer(model.levee[(scenario_index, 0)], offset_distance=-1)
    levee_buffer_top = set_precision(levee_buffer_top.buffer(0), 10**-DECIMALS)
    levee_buffer_top = model.smash_polygon(
        (scenario_index, 0), levee_buffer_top, model.get_soil_by_code("T"), status="LEVEE"
    )

    for k in range(6):
        cnstr = DStabilitySlipPlaneConstraints(
            is_zone_a_constraints_enabled=True,
            width_zone_a=constraint_width[k + 1],
            x_left_zone_a=constraint_start[k + 1],
            is_zone_b_constraints_enabled=True,
            width_zone_b=50,
            x_left_zone_b=bikr.x,
        )
        default_particle_swarm_deep = DStabilityUpliftVanParticleSwarmAnalysisMethod(
            options_type="Thorough",
            search_area_a=DStabilitySearchArea(height=20.0, top_left=Point(x=bukr.x, z=bukr.z + 20), width=30.0),
            search_area_b=DStabilitySearchArea(height=20.0, top_left=Point(x=mid_bitalud, z=bikr.z + 10), width=30),
            tangent_area_height=8.0,
            tangent_area_top_z=bit.z,
            slip_plane_constraints=cnstr,
        )
        calculations = [
            Calculation(
                label="Semi-prob (met reststerkte)",
                analysismethod=default_particle_swarm_shallow,
                calculationtype=CalculationTypeEnum.DESIGN,
            ),
            Calculation(
                label="Semi-prob (zonder reststerkte)",
                analysismethod=default_particle_swarm_deep,
                calculationtype=CalculationTypeEnum.DESIGN,
            ),
            Calculation(
                label="Probabilistisch (met reststerkte) (FORM)",
                analysismethod=default_prob,
                calculationtype=CalculationTypeEnum.PROBABILISTIC,
            ),
            Calculation(
                label="Probabilistisch (zonder reststerkte) (FORM)",
                analysismethod=default_prob,
                calculationtype=CalculationTypeEnum.PROBABILISTIC,
            ),
        ]
        if k == 4:
            calculations = [
                Calculation(
                    label="Semi-prob (met reststerkte)",
                    analysismethod=default_particle_swarm_shallow,
                    calculationtype=CalculationTypeEnum.DESIGN,
                ),
                Calculation(
                    label="Semi-prob (zonder reststerkte)",
                    analysismethod=default_particle_swarm_deep,
                    calculationtype=CalculationTypeEnum.DESIGN,
                ),
                Calculation(
                    label="Probabilistisch (met reststerkte) (FORM)",
                    analysismethod=default_prob,
                    calculationtype=CalculationTypeEnum.PROBABILISTIC,
                ),
                Calculation(
                    label="Probabilistisch (zonder reststerkte) (FORM)",
                    analysismethod=default_prob,
                    calculationtype=CalculationTypeEnum.PROBABILISTIC,
                ),
                Calculation(
                    label="Probabilistisch (zonder reststerkte) (MCIS)",
                    analysismethod=default_particle_swarm_deep,
                    calculationtype=CalculationTypeEnum.PROBABILISTIC,
                ),
            ]

        model.add_scenario((k, 0))
        if k == 5:  # stage overslag
            if indringing[0] == "Ja":
                fr_overslag = data_from_excel["FREA_OVERSLAG"]
                fr_WBN = data_from_excel["FREA_3"]
                stijg = data_from_excel["STIJG_3"]
                RL_frea_HW = RL_frea.copy
                RL_stijg_HW = RL_stijg.copy
                RL_overslag.headline_above = fr_overslag
                RL_frea_HW.headline_above = fr_WBN
                RL_stijg_HW.headline_above = stijg
                stage_overslag = Stage(
                    label="Hoogwater conditie",
                    headlines=[fr_WBN, fr_overslag, stijg, DL_stijg],
                    referencelines=[RL_frea_HW, RL_overslag, RL_stijg_HW, RL_indringing],
                    add_state=False,
                    correlate_state="within_borehole",
                    uplift=UpliftSettings(check=False),
                )
                model_config.append(
                    Scenario(
                        label="Overslag stage",
                        stages=[stage_dagelijks, stage_overslag],
                        calculations=calculations,
                        index=k + 1,
                    )
                )

            else:
                fr_overslag = data_from_excel["FREA_OVERSLAG"]
                fr_WBN = data_from_excel["FREA_3"]
                stijg = data_from_excel["STIJG_3"]
                RL_frea_HW = RL_frea.copy
                RL_stijg_HW = RL_stijg.copy
                RL_overslag.headline_above = fr_overslag
                RL_frea_HW.headline_above = fr_WBN
                RL_stijg_HW.headline_above = stijg
                stage_overslag = Stage(
                    label="Hoogwater conditie",
                    headlines=[fr_WBN, fr_overslag, stijg],
                    referencelines=[RL_frea_HW, RL_overslag, RL_stijg_HW],
                    add_state=False,
                    correlate_state="within_borehole",
                    uplift=UpliftSettings(check=False),
                )
                model_config.append(
                    Scenario(
                        label="Overslag stage",
                        stages=[stage_dagelijks, stage_overslag],
                        calculations=calculations,
                        index=k + 1,
                    )
                )

        if k < 5:
            if indringing[0] == "Ja":
                fr = data_from_excel[f"FREA_{k + 1}"]
                stijg = data_from_excel[f"STIJG_{k + 1}"]
                RL_frea_HW = RL_frea.copy
                RL_stijg_HW = RL_stijg.copy
                RL_frea_HW.headline_above = fr
                RL_stijg_HW.headline_above = stijg
                stage_hoogwater = Stage(
                    label="Hoogwater conditie",
                    headlines=[fr, stijg, DL_stijg],
                    referencelines=[RL_frea_HW, RL_stijg_HW, RL_indringing],
                    add_state=False,
                    correlate_state="within_borehole",
                    uplift=UpliftSettings(check=False),
                )
                model_config.append(
                    Scenario(
                        label="Hoogwater stage " + str(k + 1),
                        stages=[stage_dagelijks, stage_hoogwater],
                        calculations=calculations,
                        index=k + 1,
                    )
                )

            else:
                fr = data_from_excel[f"FREA_{k + 1}"]
                stijg = data_from_excel[f"STIJG_{k + 1}"]
                RL_frea_HW = RL_frea.copy
                RL_stijg_HW = RL_stijg.copy
                RL_frea_HW.headline_above = fr
                RL_stijg_HW.headline_above = stijg
                stage_hoogwater = Stage(
                    label="Hoogwater conditie",
                    headlines=[fr, stijg],
                    referencelines=[RL_frea_HW, RL_stijg_HW],
                    add_state=False,
                    correlate_state="within_borehole",
                    uplift=UpliftSettings(check=False),
                )
                model_config.append(
                    Scenario(
                        label="Hoogwater stage " + str(k + 1),
                        stages=[stage_dagelijks, stage_hoogwater],
                        calculations=calculations,
                        index=k + 1,
                    )
                )

    print(f"Creating model: {savename}")
    model.create_model(model_config, savename=savename)
    return model


def get_ids_to_remove(states_path, scenario_count, stage_count):
    """
    Verwijder POP statepoints die drained zijn.

    Scenario en stage worden automatisch bepaald.
    """
    invalid_ids = set()

    with open(states_path, encoding="utf-8") as f:
        data = json.load(f)

    for point in data.get("StatePoints", []):
        stress = point.get("Stress", {})

        if stress.get("StateType") != "Pop":
            continue

        sp_id = point.get("Id")

        if sp_id is None:
            continue

        try:
            drainage = drainage_for_statepoint(
                model.model, state_point_id=sp_id, scenario_index=scenario_count, stage_index=0, use="below"
            )

            if drainage == "drained":
                invalid_ids.add(str(sp_id))

        except Exception:
            # als drainage lookup faalt → negeren
            pass

    return invalid_ids


def clean_states_json(states_path, invalid_ids):
    """Remove invalid StatePoints"""
    with open(states_path, encoding="utf-8") as f:
        data = json.load(f)

    before = len(data.get("StatePoints", []))
    kept = []

    for point in data.get("StatePoints", []):
        if str(point.get("Id")) not in invalid_ids:
            kept.append(point)

    data["StatePoints"] = kept

    if len(kept) != before:
        with open(states_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        return True
    return False


def drainage_for_statepoint(dm: DStabilityModel, state_point_id: int, scenario_index=0, stage_index=0, use="below"):
    sp = dm._get_state(scenario_index, stage_index).get_state(state_point_id)  # PersistableStatePoint
    soil_layers = dm._get_soil_layers(scenario_index, stage_index)
    soil_id = DStabilityModel.get_soil_id_from_layer_id(soil_layers, sp.LayerId)
    soil = next(s for s in dm.soils.Soils if s.Id == soil_id)

    model = (
        soil.ShearStrengthModelTypeBelowPhreaticLevel
        if use == "below"
        else soil.ShearStrengthModelTypeAbovePhreaticLevel
    )

    if str(model) == "ShearStrengthModelTypePhreaticLevelInternal.SU":
        return "undrained"

    else:
        return "drained"


def clean_statecorrelations_json(correlations_path, invalid_ids):
    """Remove invalid IDs from CorrelatedStateIds arrays"""
    with open(correlations_path, encoding="utf-8") as f:
        data = json.load(f)

    changed = False
    for corr_group in data.get("StateCorrelations", []):
        before = len(corr_group.get("CorrelatedStateIds", []))
        valid_ids = [id for id in corr_group.get("CorrelatedStateIds", []) if id not in invalid_ids]
        corr_group["CorrelatedStateIds"] = valid_ids

        if len(valid_ids) != before:
            changed = True
            # print(f"   Cleaned correlation: {before} → {len(valid_ids)} IDs")

    if changed:
        with open(correlations_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

    return changed


def update_pop_between_x(states_path: str, x_left: float, x_right: float) -> int:
    """
    Past POP aan voor POP-statepoints met x tussen x_left en x_right:

      Mean        -> POP gem kruin
      Characteristic -> POP kar kruin
      Stdev       -> POP STDEV_prob kruin
    """
    with open(states_path, encoding="utf-8") as f:
        data = json.load(f)

    x1, x2 = min(x_left, x_right), max(x_left, x_right)
    changed = 0

    for point in data.get("StatePoints", []):
        stress = point.get("Stress", {})
        if not isinstance(stress, dict) or stress.get("StateType") != "Pop":
            continue

        x = _get_x_from_statepoint(point)
        if x is None or not (x1 <= x <= x2):
            continue

        pop_stoch = stress.get("PopStochasticParameter", {})
        if not isinstance(pop_stoch, dict):
            pop_stoch = {}
            stress["PopStochasticParameter"] = pop_stoch
        # DIT ZIT ER NU HARDCOPY IN, MOET OTD AANGEPAST WORDEN
        # Zet de 3 waarden (en wat compat keys)
        pop_stoch["Mean"] = 43
        pop_stoch["Deterministic"] = 28.713
        pop_stoch["StandardDeviation"] = 10

        changed += 1

    with open(states_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    return changed


#     return pop_kruin


def _get_x_from_statepoint(point: dict):
    for a, b in [("Point", "X"), ("Coordinate", "X"), ("Location", "X")]:
        obj = point.get(a)
        if isinstance(obj, dict) and b in obj:
            return obj[b]
    if "X" in point:
        return point["X"]
    return None

## Loop door logboek 

In [ ]:
for bestandsnaam in os.listdir(loc_templates):
    print(bestandsnaam)
    volledig_pad = os.path.join(loc_templates, bestandsnaam)

    if os.path.isfile(volledig_pad) and bestandsnaam.lower().endswith((".xls", ".xlsx", ".xlsm")):
        model = create_file(volledig_pad, bestandsnaam[0:-5])

    if os.path.isdir(volledig_pad):
        continue

    source_stix = Path(uitvoer + "\\" + bestandsnaam).with_suffix(".stix")
    base_folder = source_stix.parent

    print("FULL STATES + CORRELATIONS CLEANER + POP KRUIN UPDATE")
    print("======================================================")

    # Copy → extract
    temp_zip = source_stix.with_name("temp.zip")
    shutil.copy2(source_stix, temp_zip)

    temp_dir = base_folder / "temp_full_clean"
    if temp_dir.exists():
        shutil.rmtree(temp_dir)
    temp_dir.mkdir()

    with zipfile.ZipFile(temp_zip, "r") as z:
        z.extractall(temp_dir)

    # states files
    states_files = list(temp_dir.rglob("states*.json"))

    #  0) bepaal but/bit uit template (via SurfaceLine)
    data_from_excel = parse_excel_template(volledig_pad)
    surface = data_from_excel["SurfaceLine"]
    bit = surface.get_cpoint_by_label("binnenteen")
    but = surface.get_cpoint_by_label("buitenteen")

    # 1) POP kruin aanpassen in alle states*.json (x tussen but.x en bit.x)
    totaal_changed = 0
    for states_file in states_files:
        n = update_pop_between_x(
            states_path=str(states_file),
            x_left=but.x,
            x_right=bit.x,
        )
        totaal_changed += n
    print(f" POP kruin aangepast voor {totaal_changed} statepoints (but.x..bit.x)")

    #  2) daarna  bestaande remove-flow (drained POP verwijderen)
    all_invalid_ids = set()

    for states_file in states_files:
        stem = states_file.stem  # bijv. "states_10" of "states"
        if "_" in stem:
            stages_nummer = int(stem.split("_")[-1])
        else:
            stages_nummer = 0

        scenario_teller = (stages_nummer + 1) // 2

        invalid = get_ids_to_remove(states_file, scenario_teller, stages_nummer)
        all_invalid_ids.update(invalid)
        print(f"  {states_file.relative_to(temp_dir)}: {len(invalid)} invalid IDs")

    # Step 2: Clean states.json files
    states_changed = 0
    for states_file in states_files:
        if clean_states_json(states_file, all_invalid_ids):
            states_changed += 1

    # Step 3: Clean statecorrelations.json files
    corr_changed = 0
    corr_files = list(temp_dir.rglob("statecorrelations*.json"))
    for corr_file in corr_files:
        if clean_statecorrelations_json(corr_file, all_invalid_ids):
            corr_changed += 1

    # Step 4: Rebuild STIX
    final_stix = source_stix
    with zipfile.ZipFile(final_stix, "w", zipfile.ZIP_DEFLATED) as out_zip:
        for root, _, files in os.walk(temp_dir):
            for file in files:
                src = Path(root) / file
                arcname = str(src.relative_to(temp_dir))
                out_zip.write(src, arcname)

    # Cleanup
    temp_zip.unlink()
    shutil.rmtree(temp_dir)


print("Process completed successfully")